# 实践项目 04：XGBoost 脑疾病表格分类

本 Notebook 使用课程模拟的 `Data.csv`，每行表示一个模拟受试者记录，`label` 包含 CTL、AD、PD 和 DEP 四类。该表不来自临床采集队列，内容用于学习数据检查、预处理、XGBoost 训练、四分类评价和变量贡献分析。

Kaggle 是本项目的首选实践入口。打开公开 Notebook 后，点击“复制并编辑”保存到自己的账户，再按单元格顺序运行。下载 Notebook 到电脑运行是补充方式。

代码单元格保留行末注释，说明每一步的输入、处理和输出；参考实现与实践顺序对应。

## 任务总览

1. 核对每行的样本单位、类别数量和变量来源。
2. 处理缺失值、类别变量和标识符。
3. 补全 XGBoost 参数并完成训练。
4. 输出混淆矩阵、每类指标和预测概率。
5. 使用置换重要性分析模型依赖的变量。

## 需要保存的结果

`task4_data_summary.png`、`task4_confusion.png`、`task4_roc_pr.png`、`task4_importance.png`、`task4_result.json`。


In [ ]:
from pathlib import Path  # 导入当前步骤需要的工具
import json  # 导入当前步骤需要的工具
import numpy as np  # 导入当前步骤需要的工具
import pandas as pd  # 导入当前步骤需要的工具
import matplotlib.pyplot as plt  # 导入当前步骤需要的工具
from sklearn.model_selection import train_test_split  # 导入当前步骤需要的工具
from sklearn.compose import ColumnTransformer  # 导入当前步骤需要的工具
from sklearn.pipeline import Pipeline  # 导入当前步骤需要的工具
from sklearn.impute import SimpleImputer  # 导入当前步骤需要的工具
from sklearn.linear_model import LogisticRegression  # 导入当前步骤需要的工具
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, StandardScaler  # 导入当前步骤需要的工具
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report  # 导入当前步骤需要的工具
from sklearn.inspection import permutation_importance  # 导入当前步骤需要的工具
from xgboost import XGBClassifier  # 导入当前步骤需要的工具

SEED = 42; OUT = Path('/kaggle/working'); OUT.mkdir(exist_ok=True)  # 固定随机状态以便复现实验
DATA_PATH = None  # 保存当前步骤使用的中间结果
candidates = sorted(Path('/kaggle/input').rglob('Data.csv'))  # 读取本任务需要的数据
if DATA_PATH is None and candidates: DATA_PATH = candidates[0]  # 根据当前条件选择处理分支
assert DATA_PATH is not None, '请挂载包含课程 Data.csv 的数据集。'  # 执行当前步骤并保留结果
df = pd.read_csv(DATA_PATH)  # 读取本任务需要的数据
assert {'ID','label'}.issubset(df.columns), '需要课程 Data.csv 中的 ID 和 label 列。'  # 执行当前步骤并保留结果
y_encoder = LabelEncoder(); y = y_encoder.fit_transform(df['label'].astype(str))  # 保存当前步骤使用的中间结果
X = df.drop(columns=['ID','label'])  # 保存当前步骤使用的中间结果
print(df.shape, dict(zip(y_encoder.classes_, np.bincount(y))))  # 显示便于检查的关键信息


## 任务 1：确定样本单位和输入变量

每行对应一名受试者，`ID` 只用于识别，不作为输入。请输出类别数量、变量类型和缺失率。


In [ ]:
summary = {
    'numeric_columns': list(X.select_dtypes(include=np.number).columns),
    'categorical_columns': [column for column in X.columns if column not in X.select_dtypes(include=np.number).columns],
    'class_counts': dict(zip(y_encoder.classes_, np.bincount(y))),
    'missing_rate': df.isna().mean().round(4).to_dict(),
}
print(summary)


## 任务 2：划分数据、拟合预处理并建立基线

预处理只能使用训练集拟合。验证集用于比较逻辑回归和 XGBoost，测试集在模型设置确定后只评价一次。当前 `Data.csv` 的缺失值总数为 0；代码仍保留训练集内的填补步骤，并在下一段用一小段复制数据观察填补行为。


In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=.30, stratify=y, random_state=SEED)  # 固定随机状态以便复现实验
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=.50, stratify=y_temp, random_state=SEED)  # 固定随机状态以便复现实验
numeric_cols = list(X.select_dtypes(include=np.number).columns)  # 保存当前步骤使用的中间结果
categorical_cols = [c for c in X.columns if c not in numeric_cols]  # 保存当前步骤使用的中间结果
pre = ColumnTransformer([  # 保存当前步骤的中间结果
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), numeric_cols),  # 保存当前步骤的中间结果
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_cols),  # 保存当前步骤的中间结果
])  # 保存当前步骤使用的中间结果
baseline_pre = ColumnTransformer([  # 保存当前步骤的中间结果
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_cols),  # 保存当前步骤的中间结果
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_cols),  # 保存当前步骤的中间结果
])  # 保存当前步骤使用的中间结果
baseline = Pipeline([('pre', baseline_pre), ('clf', LogisticRegression(max_iter=3000, class_weight='balanced'))])  # 建立用于比较的模型
print(len(X_train), len(X_val), len(X_test))  # 显示便于检查的关键信息
# 用训练集复制数据制造一个缺失值，仅观察填补动作，不改动原始 Data.csv  # 保持原始数据不变
demo_col = numeric_cols[0]  # 选择第一个数值变量作为演示列
demo_values = X_train[[demo_col]].copy()  # 复制训练集中的一列
demo_values.iloc[:5, 0] = np.nan  # 在副本中放入几个缺失值
demo_imputer = SimpleImputer(strategy='median')  # 使用训练集规则建立中位数填补器
demo_imputer.fit(X_train[[demo_col]])  # 只用训练集拟合填补规则
demo_filled = demo_imputer.transform(demo_values)  # 将训练规则应用到带缺失值的副本
print('缺失演示:', demo_col, '缺失数=', int(demo_values.isna().sum().iloc[0]), '填补后缺失数=', int(np.isnan(demo_filled).sum()))  # 检查填补结果


## 任务 3：补全 XGBoost 模型

四分类模型输出四个类别的概率。


In [ ]:
model = Pipeline([('pre', pre), ('clf', XGBClassifier(
    n_estimators=180, max_depth=3, learning_rate=.05,
    objective='multi:softprob', num_class=len(y_encoder.classes_),
    subsample=.8, colsample_bytree=.8, eval_metric='mlogloss', random_state=SEED,
))])
print(model)


## 任务 4：评价四分类结果

先在验证集比较基线和 XGBoost，再在设置确定后查看测试集的混淆矩阵、每类 precision/recall/F1、宏平均 F1 和预测概率。


In [ ]:
baseline.fit(X_train, y_train)
baseline_val_pred = baseline.predict(X_val)
baseline_test_pred = baseline.predict(X_test)
baseline_val_f1 = f1_score(y_val, baseline_val_pred, average='macro')
baseline_f1 = f1_score(y_test, baseline_test_pred, average='macro')
model.fit(X_train, y_train)
val_pred = model.predict(X_val); test_prob = model.predict_proba(X_test); test_pred = test_prob.argmax(axis=1)
validation_xgboost_f1 = f1_score(y_val, val_pred, average='macro')
macro_f1 = f1_score(y_test, test_pred, average='macro')
accuracy = accuracy_score(y_test, test_pred)
cm = confusion_matrix(y_test, test_pred)
report = classification_report(y_test, test_pred, target_names=y_encoder.classes_, output_dict=True)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
axes[0].imshow(cm, cmap='Blues'); axes[0].set(xlabel='predicted', ylabel='true', title='Test confusion matrix')
axes[1].bar(y_encoder.classes_, [report[name]['f1-score'] for name in y_encoder.classes_]); axes[1].set_ylim(0, 1); axes[1].set_title('Per-class F1')
fig.tight_layout(); fig.savefig(OUT / 'task4_confusion.png', dpi=160); plt.close(fig)
print({'validation_baseline_macro_f1': baseline_val_f1, 'validation_xgboost_macro_f1': validation_xgboost_f1, 'test_macro_f1': macro_f1, 'accuracy': accuracy})


## 任务 5：变量贡献与错误分析

置换重要性描述当前模型对变量的依赖，不代表变量的生物学因果作用。


In [ ]:
pi = permutation_importance(model, X_test, y_test, n_repeats=5, random_state=SEED, scoring='f1_macro')
feature_names = np.asarray(X_test.columns)
order = np.argsort(pi.importances_mean)[-12:]
fig, axis = plt.subplots(figsize=(7, 4.5))
axis.barh(feature_names[order], pi.importances_mean[order], color='#4f7eae')
axis.set_xlabel('permutation decrease in macro F1'); axis.set_title('Variable contribution')
fig.tight_layout(); fig.savefig(OUT / 'task4_importance.png', dpi=160); plt.close(fig)
confidence = test_prob.max(axis=1)
wrong = np.where(test_pred != y_test)[0]
high_confidence_wrong = wrong[np.argsort(confidence[wrong])[::-1]][:10]
errors = [{'row_index': int(index), 'true': str(y_encoder.inverse_transform([y_test[index]])[0]), 'predicted': str(y_encoder.inverse_transform([test_pred[index]])[0]), 'confidence': float(confidence[index])} for index in high_confidence_wrong]
result = {'classes': y_encoder.classes_.tolist(), 'accuracy': float(accuracy), 'baseline_macro_f1': float(baseline_f1), 'validation_xgboost_macro_f1': float(validation_xgboost_f1), 'macro_f1': float(macro_f1), 'classification_report': report, 'high_confidence_errors': errors}
(OUT / 'task4_result.json').write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
print(errors)
